# 03 - Kafka Producer (Speed Layer)
Simulateur de flux temps reel : envoie les trajets de **Decembre** (mois 12) dans Kafka

Les mois 1-11 sont traites en batch, Decembre arrive en streaming

In [1]:
# Installer kafka-python
import subprocess
subprocess.check_call(['pip', 'install', 'kafka-python'])
print('kafka-python installe')

kafka-python installe


In [2]:
from kafka import KafkaProducer
import json, time, csv

KAFKA_BROKER = 'kafka:29092'
TOPIC = 'nyc-taxi-trips'

# Connexion au broker Kafka (optimise pour le debit)
producer = KafkaProducer(
    bootstrap_servers=KAFKA_BROKER,
    value_serializer=lambda v: json.dumps(v).encode('utf-8'),
    batch_size=32768,       # 32KB par batch Kafka
    linger_ms=5,            # Attend 5ms pour grouper les messages
    buffer_memory=67108864  # 64MB buffer
)

print(f'Producer connecte a {KAFKA_BROKER}')
print(f'Topic : {TOPIC}')

Producer connecte a kafka:29092
Topic : nyc-taxi-trips


In [ ]:
# Envoyer UNIQUEMENT les trajets de Decembre (date_id >= 336)
# Mode rafale : envoie par paquets de 200 pour un debit eleve
import datetime

CSV_PATH = '/home/jovyan/data/nyc-taxi/fact_trips_final.csv'
BATCH_SIZE = 200       # Envoyer par rafales de 200 messages
LOG_EVERY = 1000       # Log toutes les 1000 messages
DELAY = 0.05           # 50ms de pause entre chaque rafale

count = 0
skipped = 0
with open(CSV_PATH, 'r') as f:
    reader = csv.DictReader(f)
    for row in reader:
        date_id = float(row['data_id']) if row['data_id'] else None

        # Filtrer : seulement Decembre (date_id >= 336)
        if date_id is None or date_id < 336:
            skipped += 1
            continue

        # Construire le message avec timestamp
        message = {
            'date_id': date_id,
            'pickup_location_id': int(row['localizacao_partida_id']),
            'dropoff_location_id': int(row['localizacao_chegada_id']),
            'weather_id': float(row['clima_id']) if row['clima_id'] else None,
            'trip_distance': float(row['distancia_viagem']),
            'total_amount': float(row['valor_total']),
            'event_time': datetime.datetime.now().isoformat()
        }
        producer.send(TOPIC, value=message)
        count += 1

        if count % LOG_EVERY == 0:
            print(f'  {count} messages envoyes...')

        # Pause uniquement entre les rafales (pas entre chaque message)
        if count % BATCH_SIZE == 0:
            producer.flush()
            time.sleep(DELAY)

producer.flush()
print(f'\nTermine : {count} messages de Decembre envoyes dans "{TOPIC}"')
print(f'Lignes ignorees (mois 1-11) : {skipped}')

  1000 messages envoyes...
  2000 messages envoyes...
  3000 messages envoyes...
  4000 messages envoyes...
  5000 messages envoyes...
  6000 messages envoyes...
  7000 messages envoyes...
  8000 messages envoyes...
  9000 messages envoyes...
  10000 messages envoyes...
  11000 messages envoyes...
  12000 messages envoyes...
  13000 messages envoyes...
  14000 messages envoyes...
  15000 messages envoyes...
  16000 messages envoyes...
  17000 messages envoyes...
  18000 messages envoyes...
  19000 messages envoyes...
  20000 messages envoyes...
  21000 messages envoyes...
  22000 messages envoyes...
  23000 messages envoyes...
  24000 messages envoyes...
  25000 messages envoyes...
  26000 messages envoyes...
  27000 messages envoyes...
  28000 messages envoyes...
  29000 messages envoyes...
  30000 messages envoyes...
  31000 messages envoyes...
  32000 messages envoyes...
  33000 messages envoyes...
  34000 messages envoyes...
  35000 messages envoyes...
  36000 messages envoyes...
 

In [12]:
# Verifier que les messages sont bien dans Kafka
from kafka import KafkaConsumer

consumer = KafkaConsumer(
    TOPIC,
    bootstrap_servers=KAFKA_BROKER,
    auto_offset_reset='earliest',
    value_deserializer=lambda v: json.loads(v.decode('utf-8')),
    consumer_timeout_ms=5000
)

sample = []
for msg in consumer:
    sample.append(msg.value)
    if len(sample) >= 3:
        break

consumer.close()
print(f'Apercu des messages dans "{TOPIC}" :')
for s in sample:
    print(f'  {s}')

Apercu des messages dans "nyc-taxi-trips" :
  {'date_id': 336.0, 'pickup_location_id': 146, 'dropoff_location_id': 244, 'weather_id': 36.0, 'trip_distance': 3.66, 'total_amount': 25.8, 'event_time': '2026-05-28T17:32:28.400600'}
  {'date_id': 336.0, 'pickup_location_id': 146, 'dropoff_location_id': 254, 'weather_id': 36.0, 'trip_distance': 1.7, 'total_amount': 17.5, 'event_time': '2026-05-28T17:32:28.401126'}
  {'date_id': 336.0, 'pickup_location_id': 146, 'dropoff_location_id': 49, 'weather_id': 36.0, 'trip_distance': 3.85, 'total_amount': 26.0, 'event_time': '2026-05-28T17:32:28.401232'}
